# 08 - Bronze Customers

            Ingests new CSV and JSON customer files into one append-only raw
            Delta table. Missing keys and invalid timestamps are quarantined;
            contact-quality rules are applied later in Silver.

In [ ]:
%run ./05_bronze_common

In [ ]:
import uuid
from datetime import datetime, timezone

dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
dbutils.widgets.text("landing_root", "/Volumes/abc_retail/dev/landing", "Landing root")
dbutils.widgets.text("job_run_id", "manual", "Notebook or job run ID")

catalog = validate_identifier(dbutils.widgets.get("catalog").strip(), "catalog")
landing_root = dbutils.widgets.get("landing_root").strip().rstrip("/")
job_run_id = dbutils.widgets.get("job_run_id").strip()

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`bronze`")

source_name = "customers"
pipeline_name = "bronze_customers"
pipeline_run_id = str(uuid.uuid4())
start_ts = datetime.now(timezone.utc)
source_root = f"{landing_root}/{source_name}"

In [ ]:
bronze_table = f"{catalog}.bronze.bronze_customers"
quarantine_table = f"{catalog}.quarantine.customers_errors"

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`bronze`.`bronze_customers` (
        raw_payload STRING NOT NULL,
        source_format STRING NOT NULL,
        ingest_ts TIMESTAMP NOT NULL,
        source_file STRING NOT NULL,
        batch_id STRING NOT NULL,
        load_date DATE NOT NULL,
        pipeline_run_id STRING NOT NULL
    )
    USING DELTA
    PARTITIONED BY (load_date)
    TBLPROPERTIES (
        'delta.appendOnly' = 'true',
        'comment' = 'Append-only raw customer records from CSV and JSON'
    )
    """
)

In [ ]:
try:
    files_df = pending_files(catalog, source_name, source_root, {".json", ".csv"})
    file_rows = files_df.collect()
    source_paths = [row.source_file for row in file_rows]
    token, transaction_version = batch_token(files_df) if file_rows else ("none", 0)

    upsert_audit(
        catalog,
        pipeline_run_id,
        pipeline_name,
        token,
        start_ts,
        "RUNNING",
        source_paths or [source_root],
        job_run_id,
    )

    if not file_rows:
        upsert_audit(
            catalog,
            pipeline_run_id,
            pipeline_name,
            token,
            start_ts,
            "SUCCESS",
            [source_root],
            job_run_id,
            end_ts=datetime.now(timezone.utc),
            rows_read=0,
            rows_written=0,
            rows_quarantined=0,
        )
        raise SystemExit("No new customer files")

    json_paths = [path for path in source_paths if path.lower().endswith(".json")]
    csv_paths = [path for path in source_paths if path.lower().endswith(".csv")]
    frames = []

    if json_paths:
        frames.append(
            spark.read.text(json_paths).select(
                F.col("value").alias("raw_payload"),
                F.lit("json").alias("source_format"),
                F.col("_metadata.file_path").alias("source_file"),
                F.col("_metadata.file_size").alias("file_size"),
                F.col("_metadata.file_modification_time").alias("modification_time"),
            )
        )

    if csv_paths:
        csv_schema = T.StructType(
            [
                T.StructField("user_id", T.StringType(), True),
                T.StructField("email", T.StringType(), True),
                T.StructField("phone", T.StringType(), True),
                T.StructField("country", T.StringType(), True),
                T.StructField("created_at", T.StringType(), True),
                T.StructField("updated_at", T.StringType(), True),
            ]
        )
        csv_df = spark.read.option("header", "true").schema(csv_schema).csv(csv_paths)
        payload_columns = [
            F.col(column_name).alias(column_name)
            for column_name in csv_schema.fieldNames()
        ]
        frames.append(
            csv_df.select(
                F.to_json(F.struct(*payload_columns)).alias("raw_payload"),
                F.lit("csv").alias("source_format"),
                F.col("_metadata.file_path").alias("source_file"),
                F.col("_metadata.file_size").alias("file_size"),
                F.col("_metadata.file_modification_time").alias("modification_time"),
            )
        )

    raw = frames[0]
    for frame in frames[1:]:
        raw = raw.unionByName(frame)

    prepared = (
        add_lineage(raw, pipeline_run_id)
        .withColumn("json_root", F.get_json_object("raw_payload", "$"))
        .withColumn("user_id", F.get_json_object("raw_payload", "$.user_id"))
        .withColumn("created_at_raw", F.get_json_object("raw_payload", "$.created_at"))
        .withColumn("updated_at_raw", F.get_json_object("raw_payload", "$.updated_at"))
        .withColumn("created_at_check", F.expr("try_to_timestamp(created_at_raw)"))
        .withColumn("updated_at_check", F.expr("try_to_timestamp(updated_at_raw)"))
        .withColumn(
            "error_code",
            F.when(F.col("json_root").isNull(), F.lit("parse_error"))
            .when(F.col("user_id").isNull(), F.lit("missing_key"))
            .when(
                F.col("created_at_check").isNull() | F.col("updated_at_check").isNull(),
                F.lit("invalid_timestamp"),
            ),
        )
        .withColumn(
            "error_reason",
            F.when(F.col("error_code") == "parse_error", F.lit("Malformed JSON payload"))
            .when(F.col("error_code") == "missing_key", F.lit("user_id is null"))
            .when(
                F.col("error_code") == "invalid_timestamp",
                F.lit("created_at or updated_at cannot be parsed"),
            ),
        )
    )

    valid = prepared.filter(F.col("error_code").isNull()).select(
        "raw_payload",
        "source_format",
        "ingest_ts",
        "source_file",
        "batch_id",
        "load_date",
        "pipeline_run_id",
    )
    errors = prepared.filter(F.col("error_code").isNotNull()).select(
        F.sha2(
            F.concat_ws("||", "source_file", F.coalesce("user_id", F.lit("")), "raw_payload", "error_code"),
            256,
        ).alias("error_id"),
        F.lit("bronze").alias("error_stage"),
        "error_code",
        "error_reason",
        F.col("user_id").alias("record_key"),
        "raw_payload",
        F.lit(None).cast("string").alias("rescued_data"),
        "source_file",
        "batch_id",
        "load_date",
        "ingest_ts",
        F.current_date().alias("quarantine_date"),
        "pipeline_run_id",
    )

    valid_count = append_delta(
        valid, bronze_table, f"{source_name}_bronze", transaction_version
    )
    error_count = append_delta(
        errors, quarantine_table, f"{source_name}_quarantine", transaction_version
    )
    write_dq_metrics(
        catalog,
        source_name,
        pipeline_run_id,
        transaction_version,
        valid_count,
        errors.groupBy("error_code").count().collect(),
    )
    mark_files_success(catalog, source_name, files_df, pipeline_run_id)
    upsert_audit(
        catalog,
        pipeline_run_id,
        pipeline_name,
        token,
        start_ts,
        "SUCCESS",
        source_paths,
        job_run_id,
        end_ts=datetime.now(timezone.utc),
        rows_read=valid_count + error_count,
        rows_written=valid_count,
        rows_quarantined=error_count,
    )
except SystemExit as exc:
    print(str(exc))
except Exception as exc:
    upsert_audit(
        catalog,
        pipeline_run_id,
        pipeline_name,
        None,
        start_ts,
        "FAILED",
        [source_root],
        job_run_id,
        end_ts=datetime.now(timezone.utc),
        error_message=str(exc)[:4000],
    )
    raise

In [ ]:
spark.table(bronze_table).groupBy("source_format").count().display()